In [1]:
# Importing pandas for tabular data operations
# To load and manipulate the cleaned dataset
import pandas as pd
from pathlib import Path

# Loading the cleaned dataset saved during Phase 1
# To begin feature engineering without EDA helper columns
data_path = Path("../data/processed/credit_default_clean.csv")
clean_df = pd.read_csv(data_path)

print("Shape:", clean_df.shape)

Shape: (29965, 24)


In [2]:
# Selecting the six monthly bill and payment columns
# To calculate average bill and payment amounts across the observed period
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

# Calculating average monthly bill and payment amounts
# To summarise each client's historical financial activity
features_df = clean_df.copy()
features_df["average_bill_amount"] = features_df[bill_cols].mean(axis=1)
features_df["average_payment_amount"] = features_df[payment_cols].mean(axis=1)

print(features_df[["average_bill_amount", "average_payment_amount"]].head())

   average_bill_amount  average_payment_amount
0          1284.000000              114.833333
1          2846.166667              833.333333
2         16942.166667             1836.333333
3         38555.666667             1398.000000
4         18223.166667             9841.500000


In [3]:
# Summing bill and payment amounts across six months
# To compare total payment activity with total statement balances
total_bills = features_df[bill_cols].sum(axis=1)
total_payments = features_df[payment_cols].sum(axis=1)

# Calculating the payment-to-bill ratio while handling zero denominators
# To avoid infinite values when total bill amounts equal zero
features_df["payment_to_bill_ratio"] = (
    total_payments.div(total_bills.where(total_bills != 0))
)

print(features_df["payment_to_bill_ratio"].head())

0    0.089434
1    0.292791
2    0.108388
3    0.036259
4    0.540054
Name: payment_to_bill_ratio, dtype: float64


In [4]:
# Selecting the six historical repayment-status columns
# To calculate each client's maximum delay and number of delayed months
status_cols = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

# Calculating maximum positive delay and counting months with positive delays
# To summarise the severity and frequency of recorded repayment delays
features_df["maximum_payment_delay"] = features_df[status_cols].clip(lower=0).max(axis=1)
features_df["number_of_months_with_delay"] = (features_df[status_cols] > 0).sum(axis=1)

print(features_df[["maximum_payment_delay", "number_of_months_with_delay"]].head())

   maximum_payment_delay  number_of_months_with_delay
0                      2                            2
1                      2                            2
2                      0                            0
3                      0                            0
4                      0                            0


In [5]:
# Dividing the latest bill amount by the credit limit
# To measure the latest statement balance relative to available credit
features_df["latest_bill_to_credit_limit_ratio"] = (
    features_df["BILL_AMT1"] / features_df["LIMIT_BAL"]
)

print(features_df["latest_bill_to_credit_limit_ratio"].head())

0    0.195650
1    0.022350
2    0.324878
3    0.939800
4    0.172340
Name: latest_bill_to_credit_limit_ratio, dtype: float64


In [6]:
# Dividing the latest payment amount by the latest bill amount
# To compare the latest payment with the corresponding statement balance
features_df["latest_payment_to_bill_ratio"] = (
    features_df["PAY_AMT1"].div(
        features_df["BILL_AMT1"].where(features_df["BILL_AMT1"] != 0)
    )
)

print(features_df["latest_payment_to_bill_ratio"].head())

0    0.000000
1    0.000000
2    0.051917
3    0.042562
4    0.232099
Name: latest_payment_to_bill_ratio, dtype: float64


In [7]:
# Listing the seven engineered features
# To confirm that all planned features exist in the modelling dataset
feature_cols = [
    "average_bill_amount",
    "average_payment_amount",
    "payment_to_bill_ratio",
    "maximum_payment_delay",
    "number_of_months_with_delay",
    "latest_bill_to_credit_limit_ratio",
    "latest_payment_to_bill_ratio"
]

print(features_df[feature_cols].describe().T.round(3))
print("\nMissing values per feature:")
print(features_df[feature_cols].isna().sum())

                                     count       mean        std        min  \
average_bill_amount                29965.0  45029.328  63279.074 -56043.167   
average_payment_amount             29965.0   5281.297  10142.307      0.000   
payment_to_bill_ratio              29125.0      0.392      7.785   -546.929   
maximum_payment_delay              29965.0      0.682      1.074      0.000   
number_of_months_with_delay        29965.0      0.834      1.555      0.000   
latest_bill_to_credit_limit_ratio  29965.0      0.424      0.411     -0.620   
latest_payment_to_bill_ratio       27987.0     -2.387    239.285 -35436.000   

                                        25%        50%        75%         max  
average_bill_amount                4841.000  21110.833  57166.167  877313.833  
average_payment_amount             1116.667   2400.000   5600.000  627344.333  
payment_to_bill_ratio                 0.042      0.092      0.615     797.000  
maximum_payment_delay                 0.000    

In [8]:
# Counting zero and negative bill totals and latest bill amounts
# To identify denominator values causing missing or extreme ratios
total_bills = features_df[bill_cols].sum(axis=1)

print("Total bill amount:")
print("Zero:", (total_bills == 0).sum())
print("Negative:", (total_bills < 0).sum())

print("\nLatest bill amount (BILL_AMT1):")
print("Zero:", (features_df["BILL_AMT1"] == 0).sum())
print("Negative:", (features_df["BILL_AMT1"] < 0).sum())

Total bill amount:
Zero: 840
Negative: 201

Latest bill amount (BILL_AMT1):
Zero: 1978
Negative: 590


In [9]:
# Recalculating ratios only when bill denominators are positive
# To avoid misleading ratios from zero or negative bill amounts
features_df["payment_to_bill_ratio"] = (
    total_payments.div(total_bills.where(total_bills > 0))
)

features_df["latest_payment_to_bill_ratio"] = (
    features_df["PAY_AMT1"].div(
        features_df["BILL_AMT1"].where(features_df["BILL_AMT1"] > 0)
    )
)

print(features_df[[
    "payment_to_bill_ratio",
    "latest_payment_to_bill_ratio"
]].isna().sum())

payment_to_bill_ratio           1041
latest_payment_to_bill_ratio    2568
dtype: int64


In [10]:
# Summarising ratio distributions using selected percentiles
# To assess remaining extreme values without letting them dominate the summary
ratio_cols = ["payment_to_bill_ratio", "latest_payment_to_bill_ratio"]

print(features_df[ratio_cols].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T.round(3))

                                count   mean     std  min   1%    25%    50%  \
payment_to_bill_ratio         28924.0  0.486   5.607  0.0  0.0  0.043  0.095   
latest_payment_to_bill_ratio  27397.0  1.776  76.581  0.0  0.0  0.037  0.061   

                                75%     99%        max  
payment_to_bill_ratio         0.623   2.820    797.000  
latest_payment_to_bill_ratio  0.273  17.362  11453.667  


In [11]:
# Displaying the five largest values for each ratio and their bill denominators
# To identify whether extreme ratios are associated with small positive bills
print("Payment-to-bill ratio:")
print(features_df.loc[
    features_df["payment_to_bill_ratio"].nlargest(5).index,
    ["BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
     "payment_to_bill_ratio"]
].to_string(index=False))

print("\nLatest payment-to-bill ratio:")
print(features_df.loc[
    features_df["latest_payment_to_bill_ratio"].nlargest(5).index,
    ["BILL_AMT1", "PAY_AMT1", "latest_payment_to_bill_ratio"]
].to_string(index=False))

Payment-to-bill ratio:
 BILL_AMT1  BILL_AMT2  BILL_AMT3  BILL_AMT4  BILL_AMT5  BILL_AMT6  payment_to_bill_ratio
         0          0          0          0          0        200             797.000000
        -1         -1         -1         -1         -1        219             390.158879
         0          0          0          0          0        150             205.380000
      4476       2521       3529      -5364      -8943       4057             121.460145
         0          0          0          0          0       1079              83.163114

Latest payment-to-bill ratio:
 BILL_AMT1  PAY_AMT1  latest_payment_to_bill_ratio
         3     34361                  11453.666667
         3     13333                   4444.333333
        47     99007                   2106.531915
        80     90001                   1125.012500
         1      1001                   1001.000000


i will retain the two averages and two repayment-delay features, keep latest_bill_to_credit_limit_ratio, and exclude the two payment-to-bill ratios from the primary feature set. i will revisit them later if there is a clear analytical justification.

In [12]:
# Selecting the five approved engineered features
# To define a stable feature set for validation and later modelling
approved_features = [
    "average_bill_amount",
    "average_payment_amount",
    "maximum_payment_delay",
    "number_of_months_with_delay",
    "latest_bill_to_credit_limit_ratio"
]

print(features_df[approved_features].isna().sum())
print("\nShape:", features_df[approved_features].shape)

average_bill_amount                  0
average_payment_amount               0
maximum_payment_delay                0
number_of_months_with_delay          0
latest_bill_to_credit_limit_ratio    0
dtype: int64

Shape: (29965, 5)


In [13]:
# Combining the original data with the five approved engineered features
# To preserve the target and predictors in one modelling dataset
model_df = clean_df.join(features_df[approved_features])

# Saving the engineered dataset for later modelling
# To keep the cleaned source CSV unchanged
output_path = Path("../data/processed/credit_default_engineered.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
model_df.to_csv(output_path, index=False)

print("Shape:", model_df.shape)
print("Saved to:", output_path.resolve())

Shape: (29965, 29)
Saved to: D:\PYTHON\projects\credit-risk-decision-support\data\processed\credit_default_engineered.csv


In [ ]:
# Loading the engineered dataset to verify the original ratio values for SQL
import pandas as pd

df = pd.read_csv("../data/processed/credit_default_engineered.csv")

# Checking ratio values for rows with small positive bill amounts
print(
    df.loc[
        df["BILL_AMT1"].between(1, 30),
        ["LIMIT_BAL", "BILL_AMT1", "latest_bill_to_credit_limit_ratio"]
    ].head(10)
)

       LIMIT_BAL  BILL_AMT1  latest_bill_to_credit_limit_ratio
3411      160000          3                           0.000019
3814       10000         12                           0.001200
5700      360000         27                           0.000075
8324      400000          8                           0.000020
11930     360000          9                           0.000025
13962      50000          3                           0.000060
14074     320000         14                           0.000044
15352     470000          9                           0.000019
25092     360000          1                           0.000003
25337     260000         28                           0.000108
